<a href="https://colab.research.google.com/github/elsheppardo/hello-world/blob/main/Stage_5_2023_Cap_Gains_Events.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
STAGE 5: 2023 Capital Gains Events
=====================================
Purpose: Calculate the realized CAD capital gain/(loss) on every
         2023 disposition event, using:
           - CAD valuations from Stage 1 (FX rates)
           - USD ACB per $1 USD from Stage 2 (= 1.27268 CAD/USD)
           - BTC ACB from Stage 3 (running weighted average)

Output: Stage5_2023_Events.xlsx

Events covered:
  A. BTC Consolidation (Jan-Mar 2023)
     - Jan 14, 2023 Paxos sell 2 BTC
     - Mar 22, 2023 Paxos buy 2.04 BTC (for transfer to KuCoin)
     - Mar 22, 2023 KuCoin sell 2.04 BTC (main final disposition)
     - Mar 22, 2023 KuCoin buy 0.055 BTC (small residual)
     - Nov 30, 2023 small BTC disposition (residual cleanup)

  B. USD → CAD Conversions at NDAX
     - 8 EFT withdrawals Feb 2023 through Dec 2023
     - Each realizes FX gain/loss on USD disposed
     - Proceeds (CAD) = actual CAD received
     - ACB (CAD) = USD disposed × 1.27268 (ACB per USD from Stage 2)

Key output: Cell K of 2023 subtotal row = net CAD capital gain/(loss)

Expected result: meaningful capital GAIN (~$20-25k CAD) driven
primarily by FX appreciation of USD bought at avg 1.27 and converted
back at 1.35-1.38.
"""

from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.comments import Comment

FONT_NAME = "Arial"
styles = {
    'title':       Font(name=FONT_NAME, size=16, bold=True, color='1F4E79'),
    'section':     Font(name=FONT_NAME, size=12, bold=True, color='1F4E79'),
    'header':      Font(name=FONT_NAME, size=10, bold=True, color='FFFFFF'),
    'body':        Font(name=FONT_NAME, size=10),
    'body_bold':   Font(name=FONT_NAME, size=10, bold=True),
    'input':       Font(name=FONT_NAME, size=10, color='0000FF'),
    'formula':     Font(name=FONT_NAME, size=10, color='000000'),
    'link':        Font(name=FONT_NAME, size=10, color='008000'),
    'note':        Font(name=FONT_NAME, size=9,  italic=True, color='595959'),
    'header_fill':    PatternFill('solid', fgColor='1F4E79'),
    'subheader_fill': PatternFill('solid', fgColor='D5E8F0'),
    'verify_fill':    PatternFill('solid', fgColor='FFF2CC'),
    'total_fill':     PatternFill('solid', fgColor='E2EFDA'),
    'warning_fill':   PatternFill('solid', fgColor='FCE4D6'),
    'acq_fill':       PatternFill('solid', fgColor='F2F2F2'),
    'disp_fill':      PatternFill('solid', fgColor='FFF7E6'),
    'fx_fill':        PatternFill('solid', fgColor='DEEBF7'),  # light blue for USD→CAD conversions
}
thin = Side(border_style='thin', color='BFBFBF')
cell_border = Border(left=thin, right=thin, top=thin, bottom=thin)

FMT_USD    = '"$"#,##0.00;("$"#,##0.00);"-"'
FMT_CAD    = '"$"#,##0.00;("$"#,##0.00);"-"'
FMT_CRYPTO = '#,##0.00000000'
FMT_PRICE  = '"$"#,##0.00'
FMT_FX     = '0.0000'

# KEY CONSTANT: ACB per USD from Stage 2
ACB_PER_USD = 1.27268  # CAD per $1 USD (weighted avg of Jan 3 and Mar 9 fiat purchases)


def build_workbook(output_path):
    wb = Workbook()
    ws = wb.active
    ws.title = "2023_Events"

    # Column widths
    widths = [12, 14, 30, 12, 14, 14, 14, 10, 14, 14, 14, 50]
    for i, w in enumerate(widths, 1):
        ws.column_dimensions[get_column_letter(i)].width = w

    # Title
    ws.cell(row=1, column=1, value="2023 Capital Gains Events").font = styles['title']
    ws.cell(row=2, column=1, value=(
        "BTC dispositions use ACB from Stage 3. USD→CAD conversions use ACB per USD = 1.27268 (from Stage 2). "
        "FX gain on USD drives most of 2023 capital gain."
    )).font = styles['note']
    ws.cell(row=3, column=1, value=(
        f"ACB per USD (CAD): {ACB_PER_USD} — referenced for every USD disposition below."
    )).font = styles['note']

    # Headers
    headers = [
        "Date", "Platform", "Event", "Asset",
        "Amount", "Price (USD)", "USD Value", "FX Rate",
        "CAD Value", "CAD ACB", "CAD Gain/(Loss)", "Notes",
    ]
    for i, h in enumerate(headers, 1):
        cell = ws.cell(row=5, column=i, value=h)
        cell.font = styles['header']
        cell.fill = styles['header_fill']
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        cell.border = cell_border

    # Helper to write a section header
    def write_section(r, title):
        c = ws.cell(row=r, column=1, value=title)
        c.font = styles['section']
        c.fill = styles['subheader_fill']
        for col in range(1, 13):
            ws.cell(row=r, column=col).fill = styles['subheader_fill']
            ws.cell(row=r, column=col).border = cell_border

    # Helper: write an acquisition row (no gain/loss)
    def write_acquisition(r, date, platform, event, asset, amount, price_usd, fx_rate, notes,
                           usd_override=None):
        ws.cell(row=r, column=1, value=date).font = styles['body']
        ws.cell(row=r, column=2, value=platform).font = styles['body']
        ws.cell(row=r, column=3, value=event).font = styles['body']
        ws.cell(row=r, column=4, value=asset).font = styles['body']
        c = ws.cell(row=r, column=5, value=amount); c.font = styles['input']; c.number_format = FMT_CRYPTO
        c = ws.cell(row=r, column=6, value=price_usd); c.font = styles['input']; c.number_format = FMT_PRICE
        if usd_override is not None:
            c = ws.cell(row=r, column=7, value=usd_override); c.font = styles['input']
        else:
            c = ws.cell(row=r, column=7, value=f"=E{r}*F{r}"); c.font = styles['formula']
        c.number_format = FMT_USD
        c = ws.cell(row=r, column=8, value=fx_rate); c.font = styles['input']; c.fill = styles['verify_fill']; c.number_format = FMT_FX
        c = ws.cell(row=r, column=9, value=f"=G{r}*H{r}"); c.font = styles['formula']; c.number_format = FMT_CAD
        ws.cell(row=r, column=10, value="-").font = styles['body']
        ws.cell(row=r, column=10).alignment = Alignment(horizontal='center')
        ws.cell(row=r, column=11, value="-").font = styles['body']
        ws.cell(row=r, column=11).alignment = Alignment(horizontal='center')
        c = ws.cell(row=r, column=12, value=notes); c.font = styles['note']
        c.alignment = Alignment(horizontal='left', vertical='center', wrap_text=True)
        for col in range(1, 13):
            ws.cell(row=r, column=col).fill = styles['acq_fill']
            ws.cell(row=r, column=col).border = cell_border

    # Helper: write a BTC disposition row
    def write_btc_disposition(r, date, platform, event, amount, price_usd, fx_rate,
                              acb_cad, notes, usd_override=None):
        ws.cell(row=r, column=1, value=date).font = styles['body']
        ws.cell(row=r, column=2, value=platform).font = styles['body']
        ws.cell(row=r, column=3, value=event).font = styles['body']
        ws.cell(row=r, column=4, value="BTC").font = styles['body']
        c = ws.cell(row=r, column=5, value=amount); c.font = styles['input']; c.number_format = FMT_CRYPTO
        c = ws.cell(row=r, column=6, value=price_usd); c.font = styles['input']; c.number_format = FMT_PRICE
        if usd_override is not None:
            c = ws.cell(row=r, column=7, value=usd_override); c.font = styles['input']
        else:
            c = ws.cell(row=r, column=7, value=f"=E{r}*F{r}"); c.font = styles['formula']
        c.number_format = FMT_USD
        c = ws.cell(row=r, column=8, value=fx_rate); c.font = styles['input']; c.fill = styles['verify_fill']; c.number_format = FMT_FX
        c = ws.cell(row=r, column=9, value=f"=G{r}*H{r}"); c.font = styles['formula']; c.number_format = FMT_CAD
        c = ws.cell(row=r, column=10, value=acb_cad); c.font = styles['input']; c.number_format = FMT_CAD
        c = ws.cell(row=r, column=11, value=f"=I{r}-J{r}"); c.font = styles['formula']; c.number_format = FMT_CAD
        c = ws.cell(row=r, column=12, value=notes); c.font = styles['note']
        c.alignment = Alignment(horizontal='left', vertical='center', wrap_text=True)
        for col in range(1, 13):
            ws.cell(row=r, column=col).fill = styles['disp_fill']
            ws.cell(row=r, column=col).border = cell_border

    # Helper: write a USD→CAD disposition (FX conversion)
    def write_usd_to_cad(r, date, platform, event, cad_received, fx_rate, notes):
        # The actual CAD received is the anchor input (from the user's NDAX CSV).
        # USD disposed = CAD received / FX rate.
        # USD value (col G) = USD disposed (col E).
        # CAD value (col I) = actual CAD received (proceeds).
        # CAD ACB (col J) = USD disposed × ACB per USD.
        # Gain (col K) = I - J.
        ws.cell(row=r, column=1, value=date).font = styles['body']
        ws.cell(row=r, column=2, value=platform).font = styles['body']
        ws.cell(row=r, column=3, value=event).font = styles['body']
        ws.cell(row=r, column=4, value="USD").font = styles['body']
        # Amount (USD disposed) = I / H (CAD received / FX rate)
        c = ws.cell(row=r, column=5, value=f"=I{r}/H{r}"); c.font = styles['formula']; c.number_format = FMT_CRYPTO
        c = ws.cell(row=r, column=6, value=1.00); c.font = styles['input']; c.number_format = FMT_PRICE
        c = ws.cell(row=r, column=7, value=f"=E{r}"); c.font = styles['formula']; c.number_format = FMT_USD
        c = ws.cell(row=r, column=8, value=fx_rate); c.font = styles['input']; c.fill = styles['verify_fill']; c.number_format = FMT_FX
        c = ws.cell(row=r, column=9, value=cad_received); c.font = styles['input']; c.number_format = FMT_CAD
        # ACB = USD disposed × ACB per USD (constant)
        c = ws.cell(row=r, column=10, value=f"=E{r}*{ACB_PER_USD}"); c.font = styles['formula']; c.number_format = FMT_CAD
        c = ws.cell(row=r, column=11, value=f"=I{r}-J{r}"); c.font = styles['formula']; c.number_format = FMT_CAD
        c = ws.cell(row=r, column=12, value=notes); c.font = styles['note']
        c.alignment = Alignment(horizontal='left', vertical='center', wrap_text=True)
        for col in range(1, 13):
            ws.cell(row=r, column=col).fill = styles['fx_fill']
            ws.cell(row=r, column=col).border = cell_border

    # ============================================================
    # Build rows
    # ============================================================
    r = 6

    # --- Section A: BTC Consolidation ---
    write_section(r, "Section A: BTC Consolidation and Final Disposition (Jan-Mar 2023)")
    r += 1

    # Jan 14 Paxos sell 2 BTC
    # ACB per BTC at this point (from Stage 3 ledger row 15): $30,099.35
    # ACB for this sell = 2 × $30,099.35 = $60,198.69
    write_btc_disposition(
        r,
        date="2023-01-14", platform="Paxos",
        event="Sell 2 BTC (13 fills)",
        amount=2.00000000, price_usd=20807.30, fx_rate=1.3389,
        acb_cad=60198.69,
        notes="Paxos BTC sell. Proceeds from Paxos CSV. "
              "ACB from Stage 3 BTC ledger: $30,099.35/BTC × 2 BTC = $60,198.69.",
        usd_override=41598.12
    )
    r += 1

    # Mar 22 Paxos buy 2.04 BTC
    write_acquisition(
        r,
        date="2023-03-22", platform="Paxos",
        event="Buy 2.04 BTC (17 fills)", asset="BTC",
        amount=2.03796470, price_usd=28566.53, fx_rate=1.3699,
        notes="Rebuy on Paxos using available cash. To be withdrawn to KuCoin and sold.",
        usd_override=58217.23
    )
    r += 1

    # Mar 22 KuCoin sell 2.04 BTC (final main disposition)
    # ACB per BTC at this point (from Stage 3 ledger row 17): $39,110.16
    # ACB for this sell = 2.03796470 × $39,110.16 = $79,705.12
    write_btc_disposition(
        r,
        date="2023-03-22", platform="KuCoin",
        event="Sell 2.04 BTC (main disposition)",
        amount=2.03796470, price_usd=28000.00, fx_rate=1.3699,
        acb_cad=79705.12,
        notes="Transferred from Paxos to KuCoin and sold at $28,000. "
              "ACB from Stage 3 BTC ledger: $39,110.16/BTC × 2.03796470 BTC = $79,705.12."
    )
    r += 1

    # Mar 22 KuCoin buy 0.055 BTC (small rebuy)
    write_acquisition(
        r,
        date="2023-03-22", platform="KuCoin",
        event="Buy 0.055 BTC (small rebuy)", asset="BTC",
        amount=0.05485424, price_usd=27600.00, fx_rate=1.3699,
        notes="Small rebuy after main sell. Disposed later as part of late-2023 USDC tail.",
        usd_override=1513.90
    )
    r += 1

    # Nov 30 2023 small BTC disposition (residual cleanup, part of late USDC tail)
    # ACB per BTC at this point (from Stage 3 ledger row 19): $37,919.70
    # ACB for this sell = 0.05485424 × $37,919.70 = $2,080.06
    write_btc_disposition(
        r,
        date="2023-11-30", platform="KuCoin",
        event="Small BTC dispose (residual, estimated)",
        amount=0.05485424, price_usd=37230.00, fx_rate=1.3589,
        acb_cad=2080.06,
        notes="Estimated disposal of small 0.055 BTC residual. Proceeds rolled into Nov/Dec 2023 "
              "USDC withdrawals to NDAX. Nov 2023 BTC price ~$37k (VERIFY with accountant).",
        usd_override=2042.00
    )
    r += 2

    # --- Section B: USD → CAD Conversions ---
    write_section(r, "Section B: USD → CAD Conversions at NDAX (FX Gain)")
    r += 1

    # Explanatory note
    c = ws.cell(row=r, column=1, value=(
        f"Every USD→CAD conversion realizes FX gain/loss. ACB per USD = {ACB_PER_USD} CAD (from Stage 2 fiat purchases). "
        f"CAD received − (USD disposed × ACB per USD) = CAD gain/(loss)."
    ))
    c.font = styles['note']
    c.alignment = Alignment(horizontal='left', vertical='center', wrap_text=True)
    ws.merge_cells(start_row=r, start_column=1, end_row=r, end_column=12)
    for col in range(1, 13):
        ws.cell(row=r, column=col).fill = styles['warning_fill']
        ws.cell(row=r, column=col).border = cell_border
    ws.row_dimensions[r].height = 30
    r += 1

    # NDAX CAD withdrawals — each realizes FX gain/loss on USD disposed
    ndax_withdrawals = [
        ("2023-02-26", 33477.99, 1.37,
         "NDAX EFT withdrawal to bank (first of two Feb 26 withdrawals)"),
        ("2023-02-26", 56020.15, 1.37,
         "NDAX EFT withdrawal to bank (second of two Feb 26 withdrawals)"),
        ("2023-03-13", 194732.60, 1.38,
         "NDAX EFT withdrawal — largest single withdrawal"),
        ("2023-03-20", 75838.07, 1.38,
         "NDAX EFT withdrawal to bank"),
        ("2023-04-04", 19608.63, 1.35,
         "NDAX EFT withdrawal to bank"),
        ("2023-04-05", 15000.00, 1.35,
         "NDAX EFT withdrawal to bank"),
        ("2023-09-11", 12575.21, 1.35,
         "NDAX EFT withdrawal to bank"),
        ("2023-12-06", 33553.23, 1.36,
         "Final NDAX EFT withdrawal to bank"),
    ]

    for date_str, cad_amt, fx_rate, note in ndax_withdrawals:
        write_usd_to_cad(r, date_str, "NDAX", "Convert USD → CAD (EFT)",
                         cad_amt, fx_rate, note)
        r += 1
    r += 1

    # --- 2023 Subtotal ---
    last_event_row = r - 2
    ws.cell(row=r, column=1, value="2023 Capital Gains Subtotal").font = styles['body_bold']
    c = ws.cell(row=r, column=11, value=f"=SUM(K6:K{last_event_row})")
    c.font = styles['body_bold']; c.number_format = FMT_CAD
    c = ws.cell(row=r, column=12,
                value="Sum of K col. Negative values from BTC losses, positive from FX gains on USD.")
    c.font = styles['note']
    for col in range(1, 13):
        ws.cell(row=r, column=col).fill = styles['total_fill']
        ws.cell(row=r, column=col).border = cell_border
    subtotal_row = r
    r += 3

    # --- Summary ---
    ws.cell(row=r, column=1, value="Summary").font = styles['section']
    r += 1
    ws.cell(row=r, column=1, value="2023 Net Capital Gain/(Loss) in CAD:").font = styles['body_bold']
    c = ws.cell(row=r, column=5, value=f"=K{subtotal_row}")
    c.font = styles['body_bold']; c.fill = styles['total_fill']; c.number_format = FMT_CAD
    r += 1
    ws.cell(row=r, column=1, value="Taxable portion (50% inclusion rate):").font = styles['body']
    c = ws.cell(row=r, column=5, value=f"=K{subtotal_row}*0.5")
    c.font = styles['formula']; c.number_format = FMT_CAD
    ws.cell(row=r, column=6, value="(50% of net capital gain = taxable capital gain)").font = styles['note']

    # --- Reconciliation notes ---
    r += 3
    ws.cell(row=r, column=1, value="Reconciliation Check").font = styles['section']
    r += 1
    ws.cell(row=r, column=1, value="Total USD disposed (from NDAX conversions):").font = styles['body']
    # USD disposed = sum of col E for the 8 FX conversion rows (rows 15-22)
    c = ws.cell(row=r, column=5, value=f"=SUM(E15:E22)")
    c.font = styles['formula']; c.number_format = FMT_USD
    ws.cell(row=r, column=6, value="Should reconcile against NDAX USDT+USDC deposits (~$321,463 total).").font = styles['note']
    r += 1
    ws.cell(row=r, column=1, value="Total CAD received:").font = styles['body']
    c = ws.cell(row=r, column=5, value=f"=SUM(I15:I22)")
    c.font = styles['formula']; c.number_format = FMT_CAD
    ws.cell(row=r, column=6, value="Matches bank deposits from NDAX EFT.").font = styles['note']
    r += 1
    ws.cell(row=r, column=1, value="Total CAD ACB (USD disposed × ACB per USD):").font = styles['body']
    c = ws.cell(row=r, column=5, value=f"=SUM(J15:J22)")
    c.font = styles['formula']; c.number_format = FMT_CAD
    r += 1
    ws.cell(row=r, column=1, value="Net CAD FX gain on USD conversions:").font = styles['body_bold']
    c = ws.cell(row=r, column=5, value=f"=SUM(K15:K22)")
    c.font = styles['body_bold']; c.fill = styles['total_fill']; c.number_format = FMT_CAD

    ws.freeze_panes = "A6"
    wb.save(output_path)
    print(f"Saved: {output_path}")


if __name__ == "__main__":
    build_workbook("Stage5_2023_Events.xlsx")